# Uzbekistan SPI v4.0 — reproducible numerical audit for the scientific-practical report

**Purpose:** to re-verify the key SPI figures presented in the final scientific-practical report (as of 10 September 2026) against the World Bank's official `worldbank/SPI` GitHub repository at an **exact, pinned v4.0 version**, and to generate a reproducible audit trail for reviewers.

**Primary audit source**
- World Bank SPI release: **v4.0 — "202511 SPI Update"**
- Pinned commit: **`2dd02f746cb4e2159281adc74d6865f948e90ce1`**
- Per the release notes, this update incorporates **calendar-year 2024 data**.
- The audit is pinned to this exact commit, not to the `master` branch. Subsequent revisions do not change the audit's verdict.

**How to run**
1. Open the notebook in Google Colab.
2. Click **Runtime → Run all**.
3. Checks are reported as `PASS`, `WARN`, or `FAIL`.
4. `PASS` — the project figure matches the pinned World Bank value.
5. `WARN` — not a numerical error, but a causal or external claim that requires separate confirmation.
6. `FAIL` — the project figure or a recalculation does not match the pinned source.
7. The final section produces an electronic evidence package of CSV/JSON/HTML files and the raw pinned sources.

> **Methodological boundary:** the notebook verifies values and computation logic within SPI itself. For example, it verifies that `SPI.D5.2.4.CPIBY` fell from 1.0 in 2023 to 0.5 in 2024, but it cannot prove **why** this change occurred from the SPI GitHub source alone. That requires separate verification against IMF and national metadata.

## 0. Configuration, audit log, and helper functions

In [28]:
import io
import json
import hashlib
import html
import platform
import shutil
import time
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from IPython.display import display, HTML

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 240)

# -----------------------------
# Audit konfiguratsiyasi
# -----------------------------
REPO_RAW = "https://raw.githubusercontent.com/worldbank/SPI"
REPO_PAGE = "https://github.com/worldbank/SPI"
RELEASE_TAG = "v4.0"
REF_SHA = "2dd02f746cb4e2159281adc74d6865f948e90ce1"
BASE = f"{REPO_RAW}/{REF_SHA}"

COUNTRY_ISO = "UZB"
COUNTRY_NAME = "Uzbekistan"
TARGET_YEARS = [2023, 2024]
REPORT_VERSION = "10.09.2026 — final project"

EVIDENCE_DIR = Path("UZB_SPI_AUDIT_EVIDENCE")
RAW_DIR = EVIDENCE_DIR / "raw_pinned"
EVIDENCE_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

URLS = {
    "SPI_index.csv": f"{BASE}/03_output_data/SPI_index.csv",
    "SPI_data.csv": f"{BASE}/03_output_data/SPI_data.csv",
    "SPI_dimensions_sources.csv": f"{BASE}/01_raw_data/metadata/SPI_dimensions_sources.csv",
    "SPI_index_sources.csv": f"{BASE}/01_raw_data/metadata/SPI_index_sources.csv",
    "02-SPI_index.Rmd": f"{BASE}/02_programs/02-SPI_index.Rmd",
}

RUN_UTC = datetime.now(timezone.utc).isoformat()

AUDIT_LOG = []
CLAIM_AUDIT = []

def _safe(v):
    if isinstance(v, (np.floating, np.integer)):
        return v.item()
    return v

def log_check(section, check, status, observed=None, expected=None, detail=""):
    status = str(status).upper()
    if status not in {"PASS", "WARN", "FAIL"}:
        raise ValueError("status must be PASS/WARN/FAIL")
    row = {
        "Section": section,
        "Check": check,
        "Status": status,
        "Observed": _safe(observed),
        "Expected": _safe(expected),
        "Detail": detail,
    }
    AUDIT_LOG.append(row)
    icon = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌"}[status]
    print(f"{icon} {status}: {check}")
    if observed is not None or expected is not None:
        print(f"   observed={observed} | expected={expected}")
    if detail:
        print(f"   {detail}")
    return status

def check_true(section, check, condition, observed=None, expected=True, detail=""):
    return log_check(
        section, check,
        "PASS" if bool(condition) else "FAIL",
        observed=observed, expected=expected, detail=detail
    )

def check_close(section, check, actual, expected, atol=1e-10, detail=""):
    ok = (
        pd.notna(actual) and pd.notna(expected)
        and np.isclose(float(actual), float(expected), atol=atol, rtol=0)
    )
    return log_check(
        section, check, "PASS" if ok else "FAIL",
        observed=float(actual) if pd.notna(actual) else actual,
        expected=expected, detail=detail
    )

def add_claim(report_location, metric, year, expected_display, actual_raw,
              decimals, source_field, note=""):
    actual_display = (
    float(Decimal(str(actual_raw)).quantize(
        Decimal("1").scaleb(-decimals),
        rounding=ROUND_HALF_UP
    ))
    if pd.notna(actual_raw) else np.nan
)
    tol = 0.5 * (10 ** (-decimals)) + 1e-12
    ok = pd.notna(actual_display) and abs(actual_display - expected_display) <= tol
    CLAIM_AUDIT.append({
        "Report_location": report_location,
        "Metric": metric,
        "Year": year,
        "Expected_in_report": expected_display,
        "Actual_raw_WB": _safe(actual_raw),
        "Actual_at_report_precision": actual_display,
        "Decimals": decimals,
        "Difference_at_report_precision": (
            actual_display - expected_display if pd.notna(actual_display) else np.nan
        ),
        "Status": "PASS" if ok else "FAIL",
        "Pinned_source_field": source_field,
        "Note": note,
    })
    return ok

def status_style(df, status_col="Status"):
    if status_col not in df.columns:
        return df.style
    def paint(v):
        return {
            "PASS": "background-color:#e8f5e9;font-weight:600",
            "WARN": "background-color:#fff8e1;font-weight:600",
            "FAIL": "background-color:#ffebee;font-weight:700",
        }.get(str(v), "")
    return df.style.map(paint, subset=[status_col])

def decode_csv(raw_bytes):
    last_err = None
    for enc in ["utf-8-sig", "utf-8", "cp1252", "latin-1"]:
        try:
            return pd.read_csv(io.BytesIO(raw_bytes), encoding=enc), enc
        except UnicodeDecodeError as e:
            last_err = e
    raise last_err

def fetch_bytes(url, label, attempts=3, timeout=(10, 40)):
    last_error = None
    headers = {"User-Agent": "Mozilla/5.0 SPI-reproducible-audit"}
    for attempt in range(1, attempts + 1):
        try:
            r = requests.get(url, headers=headers, timeout=timeout)
            r.raise_for_status()
            data = r.content
            if not data:
                raise RuntimeError("Empty file returned")
            return data
        except Exception as e:
            last_error = e
            print(f"⚠️ {label}: {attempt}/{attempts} urinish muvaffaqiyatsiz: {e}")
            if attempt < attempts:
                time.sleep(1.5 * attempt)
    raise RuntimeError(f"{label} yuklanmadi: {last_error}")

print("Audit ishga tayyor.")
print("Pinned release:", RELEASE_TAG)
print("Pinned commit :", REF_SHA)
print("Run UTC       :", RUN_UTC)


Audit ishga tayyor.
Pinned release: v4.0
Pinned commit : 2dd02f746cb4e2159281adc74d6865f948e90ce1
Run UTC       : 2026-09-10T09:36:00.117410+00:00


## 1. Download pinned World Bank files and generate a cryptographic fingerprint (SHA-256)

In [29]:
RAW_BYTES = {}
HASH_ROWS = []

for filename, url in URLS.items():
    print(f"Yuklanmoqda: {filename}")
    data = fetch_bytes(url, filename)
    RAW_BYTES[filename] = data
    sha256 = hashlib.sha256(data).hexdigest()
    out_path = RAW_DIR / filename
    out_path.write_bytes(data)
    HASH_ROWS.append({
        "file": filename,
        "url": url,
        "bytes": len(data),
        "sha256": sha256,
        "local_copy": str(out_path),
    })

hash_manifest = pd.DataFrame(HASH_ROWS)
display(hash_manifest)

# CSV/Rmd parsing
spi, spi_encoding = decode_csv(RAW_BYTES["SPI_index.csv"])
spi_data, spi_data_encoding = decode_csv(RAW_BYTES["SPI_data.csv"])
meta, meta_encoding = decode_csv(RAW_BYTES["SPI_dimensions_sources.csv"])
index_meta, index_meta_encoding = decode_csv(RAW_BYTES["SPI_index_sources.csv"])
r_text = RAW_BYTES["02-SPI_index.Rmd"].decode("utf-8", errors="replace")

log_check("1. Sources", "Pinned fayllar muvaffaqiyatli yuklandi", "PASS",
          observed=len(RAW_BYTES), expected=len(URLS),
          detail="Barcha fayllar commit SHA orqali yuklandi, master ishlatilmadi.")

print("\nEncodinglar:")
print(" SPI_index            :", spi_encoding)
print(" SPI_data             :", spi_data_encoding)
print(" dimensions metadata  :", meta_encoding)
print(" index metadata       :", index_meta_encoding)


Yuklanmoqda: SPI_index.csv
Yuklanmoqda: SPI_data.csv
Yuklanmoqda: SPI_dimensions_sources.csv
Yuklanmoqda: SPI_index_sources.csv
Yuklanmoqda: 02-SPI_index.Rmd


,file,url,bytes,sha256,local_copy
0,SPI_index.csv,https://raw.githubusercontent.com/worldbank/SP...,1798504,9f8798ec6e466ca050994d4dc1d1c419e3e16f679c8cdf...,UZB_SPI_AUDIT_EVIDENCE/raw_pinned/SPI_index.csv
1,SPI_data.csv,https://raw.githubusercontent.com/worldbank/SP...,4405615,f72953b1be69b49a9b8c418d91adf65702bdd999875bf2...,UZB_SPI_AUDIT_EVIDENCE/raw_pinned/SPI_data.csv
2,SPI_dimensions_sources.csv,https://raw.githubusercontent.com/worldbank/SP...,39256,9b075e59ea21e6e32679a37f42e26bcd786095f1786406...,UZB_SPI_AUDIT_EVIDENCE/raw_pinned/SPI_dimensio...
3,SPI_index_sources.csv,https://raw.githubusercontent.com/worldbank/SP...,16394,0b72595aae33d64610a280b1dc9e6d16fbe3d93b729c02...,UZB_SPI_AUDIT_EVIDENCE/raw_pinned/SPI_index_so...
4,02-SPI_index.Rmd,https://raw.githubusercontent.com/worldbank/SP...,177494,0fc6f188ec0bae1e29a0a2918237b3309e83db5f3570c4...,UZB_SPI_AUDIT_EVIDENCE/raw_pinned/02-SPI_index...


✅ PASS: Pinned fayllar muvaffaqiyatli yuklandi
   observed=5 | expected=5
   Barcha fayllar commit SHA orqali yuklandi, master ishlatilmadi.

Encodinglar:
 SPI_index            : utf-8-sig
 SPI_data             : utf-8-sig
 dimensions metadata  : utf-8-sig
 index metadata       : cp1252


## 2. Source and schema checks — critical checks before the audit begins

In [30]:
CORE_COLS = [
    "country", "iso3c", "date",
    "SPI.INDEX.PIL1", "SPI.INDEX.PIL2", "SPI.INDEX.PIL3",
    "SPI.INDEX.PIL4", "SPI.INDEX.PIL5", "SPI.INDEX"
]

missing_core = [c for c in CORE_COLS if c not in spi.columns]
check_true("2. Schema", "SPI_index core columns present",
           len(missing_core) == 0, observed=missing_core, expected=[])

max_year = int(pd.to_numeric(spi["date"], errors="coerce").max())
check_close("2. Schema", "Pinned v4.0 data\u2019s latest year is 2024",
            max_year, 2024, atol=0)

dups = int(spi.duplicated(["iso3c", "date"]).sum())
check_close("2. Schema", "No iso3c+date duplicates in SPI_index",
            dups, 0, atol=0)

uzb = spi.loc[
    (spi["iso3c"] == COUNTRY_ISO) & (spi["date"].isin(TARGET_YEARS))
].sort_values("date").copy()
uzb_core = uzb[CORE_COLS].copy()

check_close("2. Schema", "UZB rows present for 2023 and 2024",
            len(uzb), 2, atol=0)

if missing_core or len(uzb) != 2:
    raise RuntimeError("Critical schema/source error. It is not safe to continue the audit.")

print("\nPinned file dimensions:")
print(" SPI_index:", spi.shape)
print(" SPI_data :", spi_data.shape)
print(" Metadata :", meta.shape)


✅ PASS: SPI_index asosiy ustunlari mavjud
   observed=[] | expected=[]
✅ PASS: Pinned v4.0 ma’lumotining oxirgi yili 2024
   observed=2024.0 | expected=2024
✅ PASS: SPI_index da iso3c+date dublikatlari yo‘q
   observed=0.0 | expected=0
✅ PASS: UZB uchun 2023 va 2024 qatorlari mavjud
   observed=2.0 | expected=2

Pinned fayl o‘lchamlari:
 SPI_index: (4557, 80)
 SPI_data : (4557, 117)
 Metadata : (103, 8)


## 3. Core evidence for the Foreword: 2023 → 2024 SPI and P1–P5

This section directly verifies the key numerical conclusion stated in the report's Foreword:
- SPI 81.99 → 81.49;
- P1 unchanged;
- P2 and P4 increased;
- P3 and P5 decreased.

In [31]:
pillar_fields = {
    "P1 — Data Use": "SPI.INDEX.PIL1",
    "P2 — Data Services": "SPI.INDEX.PIL2",
    "P3 — Data Products": "SPI.INDEX.PIL3",
    "P4 — Data Sources": "SPI.INDEX.PIL4",
    "P5 — Data Infrastructure": "SPI.INDEX.PIL5",
    "Overall SPI": "SPI.INDEX",
}

pillar_compare = (
    uzb_core.set_index("date")[list(pillar_fields.values())]
    .rename(columns={v:k for k,v in pillar_fields.items()})
    .T
)
pillar_compare.columns = ["2023", "2024"]
pillar_compare["Change_2024_minus_2023"] = pillar_compare["2024"] - pillar_compare["2023"]
display(pillar_compare.round(6))

# Foreword figures
add_claim("Foreword", "Overall SPI", 2023, 81.99,
          uzb_core.loc[uzb_core["date"]==2023, "SPI.INDEX"].iloc[0], 2, "SPI.INDEX")
add_claim("Foreword", "Overall SPI", 2024, 81.49,
          uzb_core.loc[uzb_core["date"]==2024, "SPI.INDEX"].iloc[0], 2, "SPI.INDEX")

# Foreword logical conclusions
pc = pillar_compare
check_true("3. Foreword", "Overall SPI decreased in 2024 relative to 2023",
           pc.loc["Overall SPI", "Change_2024_minus_2023"] < 0,
           observed=pc.loc["Overall SPI", "Change_2024_minus_2023"], expected="< 0")

check_true("3. Foreword", "P1 unchanged",
           abs(pc.loc["P1 — Data Use", "Change_2024_minus_2023"]) < 1e-12,
           observed=pc.loc["P1 — Data Use", "Change_2024_minus_2023"], expected=0)

check_true("3. Foreword", "P2 increased",
           pc.loc["P2 — Data Services", "Change_2024_minus_2023"] > 0,
           observed=pc.loc["P2 — Data Services", "Change_2024_minus_2023"], expected="> 0")

check_true("3. Foreword", "P3 decreased",
           pc.loc["P3 — Data Products", "Change_2024_minus_2023"] < 0,
           observed=pc.loc["P3 — Data Products", "Change_2024_minus_2023"], expected="< 0")

check_true("3. Foreword", "P4 increased",
           pc.loc["P4 — Data Sources", "Change_2024_minus_2023"] > 0,
           observed=pc.loc["P4 — Data Sources", "Change_2024_minus_2023"], expected="> 0")

check_true("3. Foreword", "P5 decreased",
           pc.loc["P5 — Data Infrastructure", "Change_2024_minus_2023"] < 0,
           observed=pc.loc["P5 — Data Infrastructure", "Change_2024_minus_2023"], expected="< 0")


,2023,2024,Change_2024_minus_2023
P1 — Data Use,90.000000,90.000000,0.00000
P2 — Data Services,76.633333,79.133333,2.50000
P3 — Data Products,85.043750,80.112500,-4.93125
P4 — Data Sources,63.283333,68.208333,4.92500
P5 — Data Infrastructure,95.000000,90.000000,-5.00000
Overall SPI,81.992083,81.490833,-0.50125


✅ PASS: Overall SPI 2024 da 2023 ga nisbatan pasaygan
   observed=-0.5012499999999989 | expected=< 0
✅ PASS: P1 o‘zgarmagan
   observed=0.0 | expected=0
✅ PASS: P2 o‘sgan
   observed=2.5 | expected=> 0
✅ PASS: P3 pasaygan
   observed=-4.931249999999977 | expected=< 0
✅ PASS: P4 o‘sgan
   observed=4.925000000000004 | expected=> 0
✅ PASS: P5 pasaygan
   observed=-5.0 | expected=< 0


'PASS'

# 4. AUTOMATED AUDIT OF THE PROJECT'S FIGURES

The sections below compare the figures recorded in the final project tables (as of 09.09.2026)
against the pinned World Bank v4.0 values.

**Interpretation:**
- `PASS` — the project figure matches the World Bank value at the precision the report itself uses.
- `FAIL` — the project figure and the pinned World Bank value do not match; the document needs correction.

### 4.1. Table 4 — Uzbekistan's SPI trend (2016–2024)

In [32]:
HISTORY_YEARS = [2016, 2018, 2020, 2021, 2022, 2023, 2024]
history_fields = {
    "SPI Umumiy ball": "SPI.INDEX",
    "P1: Data Use": "SPI.INDEX.PIL1",
    "P2: Data Services": "SPI.INDEX.PIL2",
    "P3: Data Products": "SPI.INDEX.PIL3",
    "P4: Data Sources": "SPI.INDEX.PIL4",
    "P5: Data Infrastructure": "SPI.INDEX.PIL5",
}

uzb_hist = (
    spi.loc[(spi["iso3c"] == COUNTRY_ISO) & (spi["date"].isin(HISTORY_YEARS)),
            ["date"] + list(history_fields.values())]
    .sort_values("date")
    .set_index("date")
)

history_display = uzb_hist.rename(columns={v:k for k,v in history_fields.items()}).T
display(history_display.round(4))

REPORT_HISTORY = {
    "SPI Umumiy ball": {2016:37.7, 2018:50.5, 2020:67.8, 2021:71.1, 2022:75.8, 2023:81.99, 2024:81.49},
    "P1: Data Use": {2016:60.0, 2018:60.0, 2020:70.0, 2021:70.0, 2022:90.0, 2023:90.0, 2024:90.0},
    "P2: Data Services": {2016:4.1, 2018:39.3, 2020:74.1, 2021:74.1, 2022:76.6, 2023:76.6, 2024:79.1},
    "P3: Data Products": {2016:48.3, 2018:50.0, 2020:69.1, 2021:78.7, 2022:75.7, 2023:85.0, 2024:80.1},
    "P4: Data Sources": {2016:36.2, 2018:38.4, 2020:45.9, 2021:57.5, 2022:61.6, 2023:63.3, 2024:68.2},
    "P5: Data Infrastructure": {2016:40.0, 2018:65.0, 2020:80.0, 2021:75.0, 2022:75.0, 2023:95.0, 2024:90.0},
}

for metric, vals in REPORT_HISTORY.items():
    source_field = history_fields[metric]
    for year, expected in vals.items():
        actual = uzb_hist.loc[year, source_field]
        decimals = 2 if (metric == "SPI Umumiy ball" and year in [2023, 2024]) else 1
        add_claim("Table 4", metric, year, expected, actual, decimals, source_field)


date,2016,2018,2020,2021,2022,2023,2024
SPI Umumiy ball,37.7312,50.5429,67.8033,71.0550,75.7908,81.9921,81.4908
P1: Data Use,60.0000,60.0000,70.0000,70.0000,90.0000,90.0000,90.0000
P2: Data Services,4.1000,39.3333,74.0667,74.0667,76.6333,76.6333,79.1333
P3: Data Products,48.3312,49.9562,69.0750,78.7000,75.6875,85.0437,80.1125
P4: Data Sources,36.2250,38.4250,45.8750,57.5083,61.6333,63.2833,68.2083
P5: Data Infrastructure,40.0000,65.0000,80.0000,75.0000,75.0000,95.0000,90.0000


### 4.2. Table 5 — P2 dimensions (2020, 2022, 2024)

In [33]:
P2_DIM_FIELDS = {
    "Dim 2.1: SDDS/e-GDDS": "SPI.DIM2.1.INDEX",
    "Dim 2.2: Onlayn kirish (ODIN)": "SPI.DIM2.2.INDEX",
    "Dim 2.4: NADA": "SPI.DIM2.4.INDEX",
}
P2_YEARS = [2020, 2022, 2024]

p2_table = (
    spi.loc[(spi["iso3c"] == COUNTRY_ISO) & (spi["date"].isin(P2_YEARS)),
            ["date"] + list(P2_DIM_FIELDS.values())]
    .sort_values("date")
    .set_index("date")
    .rename(columns={v:k for k,v in P2_DIM_FIELDS.items()})
    .T
)
display(p2_table.round(4))

REPORT_P2 = {
    "Dim 2.1: SDDS/e-GDDS": {2020:0.500, 2022:0.500, 2024:0.500},
    "Dim 2.2: Onlayn kirish (ODIN)": {2020:0.722, 2022:0.799, 2024:0.874},
    "Dim 2.4: NADA": {2020:1.000, 2022:1.000, 2024:1.000},
}
for metric, vals in REPORT_P2.items():
    field = P2_DIM_FIELDS[metric]
    for year, expected in vals.items():
        actual = spi.loc[(spi["iso3c"]==COUNTRY_ISO)&(spi["date"]==year), field].iloc[0]
        add_claim("Table 5", metric, year, expected, actual, 3, field)

# Hisobotdagi SDDS stsenariysi: faqat matematik sensitivity, rasmiy prognoz emas.
d22_2024 = p2_table.loc["Dim 2.2: Onlayn kirish (ODIN)", 2024]
d24_2024 = p2_table.loc["Dim 2.4: NADA", 2024]
p2_if_d21_one = (1.0 + d22_2024 + d24_2024) / 3 * 100
print(f"\nConditional sensitivity: if D2.1=1.0 and all else unchanged, P2 = {p2_if_d21_one:.2f}")
log_check("4.2 P2", "The 95-96 point SDDS result is only a conditional sensitivity calculation",
          "WARN", observed=round(p2_if_d21_one,2), expected="rasmiy prognoz emas",
          detail="World Bank GitHub kelajakdagi ballni kafolatlamaydi.")


date,2020,2022,2024
Dim 2.1: SDDS/e-GDDS,0.500,0.500,0.500
Dim 2.2: Onlayn kirish (ODIN),0.722,0.799,0.874
Dim 2.4: NADA,1.000,1.000,1.000



Shartli sensitivity: D2.1=1.0 bo‘lsa, qolganlari o‘zgarmasa P2 = 95.80
⚠️ WARN: SDDS bo‘yicha 95–96 ball natija faqat shartli sensitivity hisobidir
   observed=95.8 | expected=rasmiy prognoz emas
   World Bank GitHub kelajakdagi ballni kafolatlamaydi.


'WARN'

### 4.3. Table 6 — P3: trend across 16 SDG goals

In [34]:
P3_FIELDS = {
    "SDG 1": "SPI.D3.1.POV",
    "SDG 2": "SPI.D3.2.HNGR",
    "SDG 3": "SPI.D3.3.HLTH",
    "SDG 4": "SPI.D3.4.EDUC",
    "SDG 5": "SPI.D3.5.GEND",
    "SDG 6": "SPI.D3.6.WTRS",
    "SDG 7": "SPI.D3.7.ENRG",
    "SDG 8": "SPI.D3.8.WORK",
    "SDG 9": "SPI.D3.9.INDY",
    "SDG 10": "SPI.D3.10.NEQL",
    "SDG 11": "SPI.D3.11.CITY",
    "SDG 12": "SPI.D3.12.CNSP",
    "SDG 13": "SPI.D3.13.CLMT",
    "SDG 15": "SPI.D3.15.LAND",
    "SDG 16": "SPI.D3.16.INST",
    "SDG 17": "SPI.D3.17.PTNS",
}
P3_YEARS = [2019, 2020, 2022, 2023, 2024]

p3_table = (
    spi.loc[(spi["iso3c"] == COUNTRY_ISO) & (spi["date"].isin(P3_YEARS)),
            ["date"] + list(P3_FIELDS.values())]
    .sort_values("date")
    .set_index("date")
    .rename(columns={v:k for k,v in P3_FIELDS.items()})
    .T
)
p3_table["23→24"] = p3_table[2024] - p3_table[2023]
display(p3_table.round(4))

REPORT_P3 = {
    "SDG 1":  {2019:0.286, 2020:0.571, 2022:0.750, 2023:1.000, 2024:1.000},
    "SDG 2":  {2019:0.750, 2020:0.889, 2022:0.600, 2023:0.700, 2024:0.600},
    "SDG 3":  {2019:0.762, 2020:0.857, 2022:0.800, 2023:0.840, 2024:0.760},
    "SDG 4":  {2019:0.364, 2020:0.545, 2022:0.800, 2023:1.000, 2024:1.000},
    "SDG 5":  {2019:0.125, 2020:0.250, 2022:0.167, 2023:0.500, 2024:0.500},
    "SDG 6":  {2019:0.833, 2020:1.000, 2022:1.000, 2023:1.000, 2024:1.000},
    "SDG 7":  {2019:0.800, 2020:0.867, 2022:0.833, 2023:0.944, 2024:0.944},
    "SDG 8":  {2019:0.385, 2020:0.462, 2022:0.778, 2023:0.750, 2024:0.750},
    "SDG 9":  {2019:0.750, 2020:0.778, 2022:0.933, 2023:1.000, 2024:0.900},
    "SDG 10": {2019:0.400, 2020:0.500, 2022:0.600, 2023:0.750, 2024:0.750},
    "SDG 11": {2019:0.200, 2020:0.667, 2022:0.875, 2023:1.000, 2024:0.750},
    "SDG 12": {2019:0.500, 2020:0.500, 2022:1.000, 2023:1.000, 2024:0.800},
    "SDG 13": {2019:0.000, 2020:1.000, 2022:0.667, 2023:0.500, 2024:0.500},
    "SDG 15": {2019:0.889, 2020:1.000, 2022:0.818, 2023:0.833, 2024:0.917},
    "SDG 16": {2019:0.333, 2020:0.444, 2022:0.600, 2023:0.857, 2024:0.714},
    "SDG 17": {2019:0.643, 2020:0.722, 2022:0.889, 2023:0.933, 2024:0.933},
}
for metric, vals in REPORT_P3.items():
    field = P3_FIELDS[metric]
    for year, expected in vals.items():
        actual = spi.loc[(spi["iso3c"]==COUNTRY_ISO)&(spi["date"]==year), field].iloc[0]
        add_claim("Table 6", metric, year, expected, actual, 3, field)

p3_changed = p3_table.loc[p3_table["23→24"].abs() > 1e-12, [2023, 2024, "23→24"]].copy()
print("\nSDG goals that changed 2023\u21922024:")
display(p3_changed.round(4))


date,2019,2020,2022,2023,2024,23→24
SDG 1,0.286,0.571,0.750,1.000,1.000,0.000
SDG 2,0.750,0.889,0.600,0.700,0.600,-0.100
SDG 3,0.762,0.857,0.800,0.840,0.760,-0.080
SDG 4,0.364,0.545,0.800,1.000,1.000,0.000
SDG 5,0.125,0.250,0.167,0.500,0.500,0.000
SDG 6,0.833,1.000,1.000,1.000,1.000,0.000
SDG 7,0.800,0.867,0.833,0.944,0.944,0.000
SDG 8,0.385,0.462,0.778,0.750,0.750,0.000
SDG 9,0.750,0.778,0.933,1.000,0.900,-0.100
SDG 10,0.400,0.500,0.600,0.750,0.750,0.000



2023→2024 o‘zgargan SDG maqsadlari:


date,2023,2024,23→24
SDG 2,0.700,0.600,-0.100
SDG 3,0.840,0.760,-0.080
SDG 9,1.000,0.900,-0.100
SDG 11,1.000,0.750,-0.250
SDG 12,1.000,0.800,-0.200
SDG 15,0.833,0.917,0.084
SDG 16,0.857,0.714,-0.143


### 4.4. Table 7 — P4 dimensions

In [35]:
P4_FIELDS = {
    "4.1c: Censuses": "SPI.DIM4.1.CEN.INDEX",
    "4.1s: Surveys": "SPI.DIM4.1.SVY.INDEX",
    "4.2: Administrative (CRVS)": "SPI.DIM4.2.INDEX",
    "4.3: Geofazoviy": "SPI.DIM4.3.INDEX",
}
P4_YEARS = [2016, 2018, 2020, 2022, 2023, 2024]

p4_table = (
    spi.loc[(spi["iso3c"] == COUNTRY_ISO) & (spi["date"].isin(P4_YEARS)),
            ["date"] + list(P4_FIELDS.values())]
    .sort_values("date")
    .set_index("date")
    .rename(columns={v:k for k,v in P4_FIELDS.items()})
    .T
)
display(p4_table.round(4))

REPORT_P4 = {
    "4.1c: Censuses": {2016:0.000, 2018:0.000, 2020:0.000, 2022:0.333, 2023:0.333, 2024:0.333},
    "4.1s: Surveys": {2016:0.400, 2018:0.466, 2020:0.468, 2022:0.734, 2023:0.800, 2024:0.800},
    "4.2: Administrative (CRVS)": {2016:1.000, 2018:1.000, 2020:1.000, 2022:1.000, 2023:1.000, 2024:1.000},
    "4.3: Geofazoviy": {2016:0.049, 2018:0.071, 2020:0.367, 2022:0.398, 2023:0.398, 2024:0.595},
}
for metric, vals in REPORT_P4.items():
    field = P4_FIELDS[metric]
    for year, expected in vals.items():
        actual = spi.loc[(spi["iso3c"]==COUNTRY_ISO)&(spi["date"]==year), field].iloc[0]
        add_claim("Table 7", metric, year, expected, actual, 3, field)


date,2016,2018,2020,2022,2023,2024
4.1c: Ro‘yxatga olishlar,0.000,0.000,0.000,0.3333,0.3333,0.3333
4.1s: So‘rovlar,0.400,0.466,0.468,0.7340,0.8000,0.8000
4.2: Ma’muriy (CRVS),1.000,1.000,1.000,1.0000,1.0000,1.0000
4.3: Geofazoviy,0.049,0.071,0.367,0.3980,0.3980,0.5950


### 4.5. Table 8 — P5 / Dim 5.2 indicators

In [36]:
P5_D52 = [
    "SPI.D5.2.1.SNAU", "SPI.D5.2.2.NABY", "SPI.D5.2.3.CNIN",
    "SPI.D5.2.4.CPIBY", "SPI.D5.2.5.HOUS", "SPI.D5.2.6.EMPL",
    "SPI.D5.2.7.CGOV", "SPI.D5.2.8.FINA", "SPI.D5.2.9.MONY",
    "SPI.D5.2.10.GSBP"
]
P5_YEARS = [2022, 2023, 2024]

# Rename columns once after pivoting, to avoid duplicate 2023/2024 columns.
p5_detail = (
    spi.loc[(spi["iso3c"] == COUNTRY_ISO) & (spi["date"].isin(P5_YEARS)),
            ["date"] + P5_D52]
    .melt(id_vars="date", var_name="source_id", value_name="value")
    .pivot(index="source_id", columns="date", values="value")
    .reset_index()
)
p5_detail.columns.name = None
p5_detail["22→23"] = p5_detail[2023] - p5_detail[2022]
p5_detail["23→24"] = p5_detail[2024] - p5_detail[2023]

meta_cols = [
    c for c in [
        "source_id", "source_name", "pillar", "SPI_indicator_id",
        "spi_indicator_name", "spi_indicator_description",
        "spi_indicator_scoring", "websites"
    ] if c in meta.columns
]
p5_detail = p5_detail.merge(meta[meta_cols].drop_duplicates("source_id"), on="source_id", how="left")
display(p5_detail.round(4))

REPORT_P5 = {
    "SPI.D5.2.1.SNAU":  {2022:0.500, 2023:1.000, 2024:1.000},
    "SPI.D5.2.2.NABY":  {2022:1.000, 2023:1.000, 2024:1.000},
    "SPI.D5.2.3.CNIN":  {2022:1.000, 2023:1.000, 2024:1.000},
    "SPI.D5.2.4.CPIBY": {2022:0.500, 2023:1.000, 2024:0.500},
    "SPI.D5.2.5.HOUS":  {2022:1.000, 2023:1.000, 2024:1.000},
    "SPI.D5.2.6.EMPL":  {2022:1.000, 2023:1.000, 2024:1.000},
    "SPI.D5.2.7.CGOV":  {2022:0.500, 2023:0.500, 2024:0.500},
    "SPI.D5.2.8.FINA":  {2022:1.000, 2023:1.000, 2024:1.000},
    "SPI.D5.2.9.MONY":  {2022:1.000, 2023:1.000, 2024:1.000},
    "SPI.D5.2.10.GSBP": {2022:0.000, 2023:1.000, 2024:1.000},
}
for field, vals in REPORT_P5.items():
    for year, expected in vals.items():
        actual = spi.loc[(spi["iso3c"]==COUNTRY_ISO)&(spi["date"]==year), field].iloc[0]
        add_claim("Table 8", field, year, expected, actual, 3, field)

cpi_2023 = spi.loc[(spi["iso3c"]==COUNTRY_ISO)&(spi["date"]==2023), "SPI.D5.2.4.CPIBY"].iloc[0]
cpi_2024 = spi.loc[(spi["iso3c"]==COUNTRY_ISO)&(spi["date"]==2024), "SPI.D5.2.4.CPIBY"].iloc[0]
check_true("4.5 P5", "The CPIBY 1.0\u21920.5 change is present in P5\u2019s mathematical decline",
           np.isclose(cpi_2023,1.0) and np.isclose(cpi_2024,0.5),
           observed=f"{cpi_2023} → {cpi_2024}", expected="1.0 → 0.5")

log_check("4.5 P5", "Why CPIBY is 0.5 cannot be proven from SPI GitHub alone",
          "WARN", observed="score change confirmed", expected="cause requires IMF/NSC metadata",
          detail="The report should present the cause as a [Hypothesis] or support it with an additional source.")


,source_id,2022,2023,2024,22→23,23→24,source_name,pillar,SPI_indicator_id,spi_indicator_name,spi_indicator_description,spi_indicator_scoring,websites
0,SPI.D5.2.1.SNAU,0.5,1.0,1.0,0.5,0.0,System of national accounts in use,Pillar 5: Data Infrastructure,Dimension 5.2,standards,The national accounts data are compiled using ...,Scoring: 1 point for using SNA2008 or ESA 2010...,http://data.worldbank.org/products/wdi
1,SPI.D5.2.10.GSBP,0.0,1.0,1.0,1.0,0.0,Business process,Pillar 5: Data Infrastructure,Dimension 5.2,standards,The Generic Statistical Business Process Model...,1 Point. GSBPM is in use. 0 Points. Otherwise,https://statswiki.unece.org/display/GSBPM/Unit...
2,SPI.D5.2.2.NABY,1.0,1.0,1.0,0.0,0.0,National Accounts base year,Pillar 5: Data Infrastructure,Dimension 5.2,standards,National accounts base year is the year used a...,"1 point for chained price, 0.5 for reference p...",http://data.worldbank.org/products/wdi
3,SPI.D5.2.3.CNIN,1.0,1.0,1.0,0.0,0.0,Classification of national industry,Pillar 5: Data Infrastructure,Dimension 5.2,standards,The industrial production data are compiled us...,1 Point. Latest version is adopted (ISIC Rev 4...,https://unstats.un.org/UNSD/mbs/app/DataSearch...
4,SPI.D5.2.4.CPIBY,0.5,1.0,0.5,0.5,-0.5,CPI base year,Pillar 5: Data Infrastructure,Dimension 5.2,standards,Consumer Price Index serves as indicators of i...,1 Point. Annual chain linking. 0.5 Points. Bas...,http://www.elibrary.imf.org/browse?freeFilter=...
5,SPI.D5.2.5.HOUS,1.0,1.0,1.0,0.0,0.0,Classification of household consumption,Pillar 5: Data Infrastructure,Dimension 5.2,standards,Classification of Individual Consumption Accor...,1 Point. Follow Classification of Individual C...,http://dsbb.imf.org/Default.aspx
6,SPI.D5.2.6.EMPL,1.0,1.0,1.0,0.0,0.0,Classification of status of employment,Pillar 5: Data Infrastructure,Dimension 5.2,standards,Classification of status of employment refers ...,1 Point. Follow International Labour Organizat...,http://dsbb.imf.org/Default.aspx
7,SPI.D5.2.7.CGOV,0.5,0.5,0.5,0.0,0.0,Central government accounting status,Pillar 5: Data Infrastructure,Dimension 5.2,standards,Government finance accounting status refers to...,1 Point. Consolidated central government accou...,http://www.elibrary.imf.org/browse?freeFilter=...
8,SPI.D5.2.8.FINA,1.0,1.0,1.0,0.0,0.0,Compilation of government finance statistics,Pillar 5: Data Infrastructure,Dimension 5.2,standards,Compilation of government finance statistics r...,1 Point. Follow the latest Government Finance ...,http://dsbb.imf.org/Default.aspx
9,SPI.D5.2.9.MONY,1.0,1.0,1.0,0.0,0.0,Compilation of monetary and financial statistics,Pillar 5: Data Infrastructure,Dimension 5.2,standards,Compilation of monetary and financial statisti...,1 Point. Follow the latest Monetary and Financ...,http://www.elibrary.imf.org/browse?freeFilter=...


✅ PASS: P5 matematik pasayishida CPIBY 1.0→0.5 o‘zgarishi mavjud
   observed=1.0 → 0.5 | expected=1.0 → 0.5
⚠️ WARN: CPIBY nima sababdan 0.5 bo‘lgani SPI GitHub bilan yolg‘iz isbotlanmaydi
   observed=score change confirmed | expected=cause requires IMF/NSC metadata
   Hujjatda sababni [Gipoteza] yoki qo‘shimcha manba bilan berish kerak.


'WARN'

### 4.6. Table 9 — 2024 international comparison and global ranks

In [38]:
COMPARATORS = {
    "Norway": {"iso3c":"NOR", "overall":94.14, "p1":100.0, "p2":98.6, "p3":84.0, "p4":88.1, "p5":100.0, "rank":1,  "chg":-0.4},
    "Korea, Rep.": {"iso3c":"KOR", "overall":91.57, "p1":100.0, "p2":96.1, "p3":87.7, "p4":79.0, "p5":95.0, "rank":7,  "chg":0.1},
    "Finland": {"iso3c":"FIN", "overall":90.37, "p1":100.0, "p2":98.2, "p3":81.6, "p4":87.1, "p5":85.0, "rank":16, "chg":-4.7},
    "Georgia": {"iso3c":"GEO", "overall":87.72, "p1":100.0, "p2":93.8, "p3":83.7, "p4":81.1, "p5":80.0, "rank":32, "chg":-1.6},
    "Moldova": {"iso3c":"MDA", "overall":85.67, "p1":100.0, "p2":96.0, "p3":79.5, "p4":67.9, "p5":85.0, "rank":38, "chg":-0.4},
    "Kazakhstan": {"iso3c":"KAZ", "overall":84.89, "p1":100.0, "p2":90.1, "p3":79.2, "p4":80.2, "p5":75.0, "rank":44, "chg":2.1},
    "Armenia": {"iso3c":"ARM", "overall":82.18, "p1":90.0, "p2":86.2, "p3":82.8, "p4":71.9, "p5":80.0, "rank":55, "chg":-1.2},
    "Uzbekistan": {"iso3c":"UZB", "overall":81.49, "p1":90.0, "p2":79.1, "p3":80.1, "p4":68.2, "p5":90.0, "rank":58, "chg":-0.5},
}

rank_2024 = spi.loc[(spi["date"]==2024) & spi["SPI.INDEX"].notna(),
                    ["country","iso3c","SPI.INDEX",
                     "SPI.INDEX.PIL1","SPI.INDEX.PIL2","SPI.INDEX.PIL3",
                     "SPI.INDEX.PIL4","SPI.INDEX.PIL5"]].copy()
rank_2024["Global_rank"] = rank_2024["SPI.INDEX"].rank(method="min", ascending=False).astype(int)

n_ranked = len(rank_2024)
check_close("4.6 International", "Number of economies with a 2024 overall SPI value",
            n_ranked, 188, atol=0,
            detail="Verifies the report\u2019s claim of 188 economies.")

rows = []
for label, exp in COMPARATORS.items():
    iso = exp["iso3c"]
    r24 = rank_2024.loc[rank_2024["iso3c"]==iso]
    r23 = spi.loc[(spi["iso3c"]==iso)&(spi["date"]==2023), ["SPI.INDEX"]]
    if len(r24)!=1 or len(r23)!=1:
        log_check("4.6 International", f"{label} row present", "FAIL",
                  observed=f"2024 rows={len(r24)}, 2023 rows={len(r23)}", expected="1 and 1")
        continue
    r24 = r24.iloc[0]
    chg = float(r24["SPI.INDEX"] - r23.iloc[0]["SPI.INDEX"])
    rows.append({
        "Country": label, "iso3c": iso,
        "Overall": r24["SPI.INDEX"],
        "P1": r24["SPI.INDEX.PIL1"],
        "P2": r24["SPI.INDEX.PIL2"],
        "P3": r24["SPI.INDEX.PIL3"],
        "P4": r24["SPI.INDEX.PIL4"],
        "P5": r24["SPI.INDEX.PIL5"],
        "Global_rank": int(r24["Global_rank"]),
        "23→24": chg,
    })
    add_claim("Table 9", f"{label} — Overall", 2024, exp["overall"], r24["SPI.INDEX"], 2, "SPI.INDEX")
    for key, field in [("p1","SPI.INDEX.PIL1"),("p2","SPI.INDEX.PIL2"),("p3","SPI.INDEX.PIL3"),
                       ("p4","SPI.INDEX.PIL4"),("p5","SPI.INDEX.PIL5")]:
        add_claim("Table 9", f"{label} — {key.upper()}", 2024, exp[key], r24[field], 1, field)
    add_claim("Table 9", f"{label} — Global rank", 2024, exp["rank"], int(r24["Global_rank"]), 0, "rank(SPI.INDEX)")
    add_claim("Table 9", f"{label} — 23→24", 2024, exp["chg"], chg, 1, "SPI.INDEX difference")

international_table = pd.DataFrame(rows)
display(international_table.round(4))


✅ PASS: 2024 SPI overall mavjud iqtisodiyotlar soni
   observed=188.0 | expected=188
   Hisobotdagi 188 ta iqtisodiyot da’vosini tekshiradi.


,Country,iso3c,Overall,P1,P2,P3,P4,P5,Global_rank,23→24
0,Norway,NOR,94.1433,100.0,98.5667,84.0000,88.1500,100.0,1,-0.4133
1,"Korea, Rep.",KOR,91.5675,100.0,96.1000,87.7125,79.0250,95.0,7,0.1150
2,Finland,FIN,90.3713,100.0,98.2000,81.5562,87.1000,85.0,16,-4.7442
3,Georgia,GEO,87.7171,100.0,93.7667,83.7188,81.1000,80.0,32,-1.5504
4,Moldova,MDA,85.6742,100.0,96.0000,79.4625,67.9083,85.0,38,-0.3975
5,Kazakhstan,KAZ,84.8933,100.0,90.1333,79.1750,80.1583,75.0,44,2.0879
6,Armenia,ARM,82.1804,90.0,86.2000,82.7688,71.9333,80.0,55,-1.1842
7,Uzbekistan,UZB,81.4908,90.0,79.1333,80.1125,68.2083,90.0,58,-0.5012


## 5. PASS/FAIL table for all project figures

In [42]:
claims_audit_df = pd.DataFrame(CLAIM_AUDIT)
if claims_audit_df.empty:
    raise RuntimeError("CLAIM_AUDIT is empty \u2014 no project figures have been checked.")

claim_counts = claims_audit_df["Status"].value_counts().reindex(["PASS","FAIL"], fill_value=0)
print("Project numerical claims:")
print(claim_counts.to_string())

fails = claims_audit_df.loc[claims_audit_df["Status"]=="FAIL"].copy()
if len(fails):
    display(HTML("<h3 style='color:#b71c1c'>❌ LOYIHADA WORLD BANK v4.0 BILAN MOS KELMAGAN RAQAMLAR TOPILDI</h3>"))
    display(status_style(fails))
else:
    display(HTML("<h3 style='color:#1b5e20'>✅ Tekshirilgan loyiha SPI raqamlari pinned World Bank v4.0 bilan mos</h3>"))

display(status_style(claims_audit_df))


Loyiha raqamli claimlari:
Status
PASS    313
FAIL      2


,Report_location,Metric,Year,Expected_in_report,Actual_raw_WB,Actual_at_report_precision,Decimals,Difference_at_report_precision,Status,Pinned_source_field,Note
191,9-jadval,Norway — P4,2024,88.200000,88.150000,88.100000,1,-0.100000,FAIL,SPI.INDEX.PIL4,
255,9-jadval,Norway — P4,2024,88.200000,88.150000,88.100000,1,-0.100000,FAIL,SPI.INDEX.PIL4,


,Report_location,Metric,Year,Expected_in_report,Actual_raw_WB,Actual_at_report_precision,Decimals,Difference_at_report_precision,Status,Pinned_source_field,Note
0,So‘zboshi,Overall SPI,2023,81.990000,81.992083,81.990000,2,0.000000,PASS,SPI.INDEX,
1,So‘zboshi,Overall SPI,2024,81.490000,81.490833,81.490000,2,0.000000,PASS,SPI.INDEX,
2,4-jadval,SPI Umumiy ball,2016,37.700000,37.731250,37.700000,1,0.000000,PASS,SPI.INDEX,
3,4-jadval,SPI Umumiy ball,2018,50.500000,50.542917,50.500000,1,0.000000,PASS,SPI.INDEX,
4,4-jadval,SPI Umumiy ball,2020,67.800000,67.803333,67.800000,1,0.000000,PASS,SPI.INDEX,
5,4-jadval,SPI Umumiy ball,2021,71.100000,71.055000,71.100000,1,0.000000,PASS,SPI.INDEX,
6,4-jadval,SPI Umumiy ball,2022,75.800000,75.790833,75.800000,1,0.000000,PASS,SPI.INDEX,
7,4-jadval,SPI Umumiy ball,2023,81.990000,81.992083,81.990000,2,0.000000,PASS,SPI.INDEX,
8,4-jadval,SPI Umumiy ball,2024,81.490000,81.490833,81.490000,2,0.000000,PASS,SPI.INDEX,
9,4-jadval,P1: Data Use,2016,60.000000,60.000000,60.000000,1,0.000000,PASS,SPI.INDEX.PIL1,


## 6. Overall SPI formula and the contribution of the five pillars

In [43]:
check_overall = uzb_core[[
    "date",
    "SPI.INDEX.PIL1", "SPI.INDEX.PIL2", "SPI.INDEX.PIL3",
    "SPI.INDEX.PIL4", "SPI.INDEX.PIL5", "SPI.INDEX"
]].copy()

pillar_cols = [f"SPI.INDEX.PIL{i}" for i in range(1,6)]
check_overall["Python_recalc"] = check_overall[pillar_cols].mean(axis=1)
check_overall["Farq"] = check_overall["Python_recalc"] - check_overall["SPI.INDEX"]
display(check_overall.round(12))

max_overall_diff = float(check_overall["Farq"].abs().max())
check_close("6. Formula", "Overall SPI = simple arithmetic mean of P1\u2013P5",
            max_overall_diff, 0.0, atol=1e-10)

contrib = pillar_compare.drop(index="Overall SPI").copy()
contrib["Overall_SPI_ga_hissa"] = contrib["Change_2024_minus_2023"] / 5
display(contrib.round(6))

overall_change = pillar_compare.loc["Overall SPI", "Change_2024_minus_2023"]
contrib_sum = contrib["Overall_SPI_ga_hissa"].sum()
check_close("6. Formula", "Sum of pillar contributions equals the change in overall SPI",
            contrib_sum, overall_change, atol=1e-10)


,date,SPI.INDEX.PIL1,SPI.INDEX.PIL2,SPI.INDEX.PIL3,SPI.INDEX.PIL4,SPI.INDEX.PIL5,SPI.INDEX,Python_recalc,Farq
269,2023,90.0,76.633333,85.04375,63.283333,95.0,81.992083,81.992083,0.0
57,2024,90.0,79.133333,80.11250,68.208333,90.0,81.490833,81.490833,-0.0


✅ PASS: Overall SPI = P1–P5 oddiy arifmetik o‘rtachasi
   observed=1.4210854715202004e-14 | expected=0.0


,2023,2024,Change_2024_minus_2023,Overall_SPI_ga_hissa
P1 — Data Use,90.000000,90.000000,0.00000,0.00000
P2 — Data Services,76.633333,79.133333,2.50000,0.50000
P3 — Data Products,85.043750,80.112500,-4.93125,-0.98625
P4 — Data Sources,63.283333,68.208333,4.92500,0.98500
P5 — Data Infrastructure,95.000000,90.000000,-5.00000,-1.00000


✅ PASS: Pillar hissalari yig‘indisi overall SPI o‘zgarishiga teng
   observed=-0.5012499999999945 | expected=-0.5012499999999989


'PASS'

## 7. World Bank missing-data process: LOCF check

In [44]:
processed_indicator_cols = [
    c for c in spi_data.columns
    if c.startswith("SPI.D") and not c.startswith("SPI.DIM")
]
final_indicator_cols = [
    c for c in spi.columns
    if c.startswith("SPI.D") and not c.startswith("SPI.DIM")
]
common_cols = sorted(set(processed_indicator_cols).intersection(final_indicator_cols))

uzb_processed = spi_data.loc[spi_data["iso3c"]==COUNTRY_ISO].sort_values("date").copy()
uzb_processed_ffill = uzb_processed.copy()
uzb_processed_ffill[processed_indicator_cols] = uzb_processed_ffill[processed_indicator_cols].ffill()

left = (
    uzb_processed_ffill.loc[uzb_processed_ffill["date"].isin(TARGET_YEARS),
                            ["date"] + common_cols]
    .set_index("date").sort_index()
)
right = (
    spi.loc[(spi["iso3c"]==COUNTRY_ISO)&(spi["date"].isin(TARGET_YEARS)),
            ["date"] + common_cols]
    .set_index("date").sort_index()
)

locf_diff = (left - right).abs()
arr = locf_diff.to_numpy(dtype=float)
max_locf_diff = float(np.nanmax(arr)) if np.isfinite(arr).any() else np.nan
print("LOCF'dan keyingi maksimal farq:", max_locf_diff)

locf_bad_cols = [
    c for c in common_cols
    if not np.allclose(left[c], right[c], equal_nan=True, atol=1e-12)
]
check_true("7. LOCF", "UZB 2023/2024 LOCF natijalari SPI_index bilan mos",
           len(locf_bad_cols)==0, observed=locf_bad_cols, expected=[])


LOCF'dan keyingi maksimal farq: 1.1102230246251565e-16
✅ PASS: UZB 2023/2024 LOCF natijalari SPI_index bilan mos
   observed=[] | expected=[]


'PASS'

## 8. Dimensions and individual indicators — full audit trail

In [45]:
dim_cols = [c for c in spi.columns if c.startswith("SPI.DIM") and c.endswith(".INDEX")]

dims = (
    uzb[["date"] + dim_cols]
    .melt(id_vars="date", var_name="dimension_code", value_name="value")
    .pivot(index="dimension_code", columns="date", values="value")
    .reset_index()
)
dims.columns.name = None
dims = dims.rename(columns={2023:"2023_raw", 2024:"2024_raw"})
dims["2023"] = dims["2023_raw"] * 100
dims["2024"] = dims["2024_raw"] * 100
dims["Change"] = dims["2024"] - dims["2023"]
dims = dims[["dimension_code","2023","2024","Change"]]
display(dims.round(4))

indicator_cols = [
    c for c in spi.columns
    if c.startswith("SPI.D") and not c.startswith("SPI.DIM")
]
ind_long = uzb[["date"] + indicator_cols].melt(
    id_vars="date", var_name="source_id", value_name="value"
)
ind_compare = ind_long.pivot(index="source_id", columns="date", values="value").reset_index()
ind_compare.columns.name = None
ind_compare = ind_compare.rename(columns={2023:"2023_raw", 2024:"2024_raw"})
ind_compare["2023"] = ind_compare["2023_raw"] * 100
ind_compare["2024"] = ind_compare["2024_raw"] * 100
ind_compare["Change"] = ind_compare["2024"] - ind_compare["2023"]
ind_compare["Abs_change"] = ind_compare["Change"].abs()
ind_compare = ind_compare.drop(columns=["2023_raw","2024_raw"])

ind_compare = ind_compare.merge(
    meta[meta_cols].drop_duplicates("source_id"),
    on="source_id", how="left"
)
changed = ind_compare.loc[ind_compare["Abs_change"].fillna(0) > 1e-9].sort_values(
    "Abs_change", ascending=False
).copy()

print("Barcha individual indikatorlar:")
display(ind_compare.sort_values("source_id").round(4))
print("\nIndicators that changed 2023\u21922024:")
display(changed.round(4))


,dimension_code,2023,2024,Change
0,SPI.DIM1.5.INDEX,90.0000,90.0000,0.0000
1,SPI.DIM2.1.INDEX,50.0000,50.0000,0.0000
2,SPI.DIM2.2.INDEX,79.9000,87.4000,7.5000
3,SPI.DIM2.4.INDEX,100.0000,100.0000,0.0000
4,SPI.DIM3.1.INDEX,84.0000,81.0000,-3.0000
5,SPI.DIM3.2.INDEX,90.7333,81.5667,-9.1667
6,SPI.DIM3.3.INDEX,66.6500,70.8500,4.2000
7,SPI.DIM3.4.INDEX,89.5000,82.3500,-7.1500
8,SPI.DIM4.1.CEN.INDEX,33.3333,33.3333,0.0000
9,SPI.DIM4.1.SVY.INDEX,80.0000,80.0000,0.0000


Barcha individual indikatorlar:


,source_id,2023,2024,Change,Abs_change,source_name,pillar,SPI_indicator_id,spi_indicator_name,spi_indicator_description,spi_indicator_scoring,websites
0,SPI.D1.5.CHLD.MORT,100.0,100.0,0.0,0.0,"Availability of Mortality rate, under-5 (per 1...",Pillar 1: Data Use,Dimension 1.5,Data use by international organisations,Child Mortality Metadata from UN IGME,1 Point. At least three indicators that met UN...,https://childmortality.org/data/
1,SPI.D1.5.DT.TDS.DPPF.XP.ZS,100.0,100.0,0.0,0.0,Quality of Debt service data according to Worl...,Pillar 1: Data Use,Dimension 1.5,Data use by international organisations,Debt Reporting Metadata from World Bank,1 Points. Actual value. 0.67 Points. Prelimina...,http://api.worldbank.org/v2/sources/2/country/...
2,SPI.D1.5.LFP,50.0,50.0,0.0,0.0,Labor force participation rate by sex and age (%),Pillar 1: Data Use,Dimension 1.5,Data use by international organisations,Labor force participation data for use by ILO,1 Point. Country has a labor force survey base...,https://www.ilo.org/shinyapps/bulkexplorer2/?l...
3,SPI.D1.5.POV,100.0,100.0,0.0,0.0,Availability of Comparable Poverty headcount r...,Pillar 1: Data Use,Dimension 1.5,Data use by international organisations,Comparability data from World Bank's Povcalnet,1 Point. Comparable data lasting at least two ...,https://development-data-hub-s3-public.s3.amaz...
4,SPI.D1.5.SAFE.MAN.WATER,100.0,100.0,0.0,0.0,Safely Managed Drinking Water,Pillar 1: Data Use,Dimension 1.5,Data use by international organisations,Availability of Safely Managed Drinking Water ...,"1 Point. At least two estimates, with breakdow...",https://washdata.org/data/country/AFG/househol...
5,SPI.D2.1.GDDS,50.0,50.0,0.0,0.0,SDDS/e-GDDS subscription,Pillar 2: Data Services,Dimension 2.1,Data releases,The Special Data Dissemination Standard (SDDS)...,Point. Subscribing to IMF SDDS+ or SDDS standa...,http://api.worldbank.org/v2/country/all/indica...
6,SPI.D2.2.Download.options,55.7,76.9,21.2,21.2,Download Options Score,Pillar 2: Data Services,Dimension 2.2,Online access,Download Options Score,Our source for this indicator is Open Data Wat...,https://odin.opendatawatch.com/
7,SPI.D2.2.Machine.readable,100.0,100.0,0.0,0.0,Machine Readability Score,Pillar 2: Data Services,Dimension 2.2,Online access,Machine Readability Score,Our source for this indicator is Open Data Wat...,https://odin.opendatawatch.com/
8,SPI.D2.2.Metadata.available,48.3,67.1,18.8,18.8,Metadata Available Score,Pillar 2: Data Services,Dimension 2.2,Online access,Metadata Available Score,Our source for this indicator is Open Data Wat...,https://odin.opendatawatch.com/
9,SPI.D2.2.Non.proprietary,100.0,100.0,0.0,0.0,Non-Proprietary format Score,Pillar 2: Data Services,Dimension 2.2,Online access,Non-Proprietary format Score,Our source for this indicator is Open Data Wat...,https://odin.opendatawatch.com/



2023→2024 o‘zgargan indikatorlar:


,source_id,2023,2024,Change,Abs_change,source_name,pillar,SPI_indicator_id,spi_indicator_name,spi_indicator_description,spi_indicator_scoring,websites
45,SPI.D5.2.4.CPIBY,100.0,50.0,-50.0,50.0,CPI base year,Pillar 5: Data Infrastructure,Dimension 5.2,standards,Consumer Price Index serves as indicators of i...,1 Point. Annual chain linking. 0.5 Points. Bas...,http://www.elibrary.imf.org/browse?freeFilter=...
15,SPI.D3.11.CITY,100.0,75.0,-25.0,25.0,GOAL 11: Sustainable Cities and Communities,Pillar 3: Data Products,Dimension 3.11,SDG Goal 11,SDG Goal 11 data availability. Source: UN Glo...,Fraction of Indicators in Goal 11 with value p...,https://unstats.un.org/sdgs/unsdg
6,SPI.D2.2.Download.options,55.7,76.9,21.2,21.2,Download Options Score,Pillar 2: Data Services,Dimension 2.2,Online access,Download Options Score,Our source for this indicator is Open Data Wat...,https://odin.opendatawatch.com/
16,SPI.D3.12.CNSP,100.0,80.0,-20.0,20.0,GOAL 12: Responsible Consumption and Production,Pillar 3: Data Products,Dimension 3.12,SDG Goal 12,SDG Goal 12 data availability. Source: UN Glo...,Fraction of Indicators in Goal 12 with value p...,https://unstats.un.org/sdgs/unsdg
39,SPI.D4.3.GEO.first.admin.level,39.8,59.5,19.7,19.7,Geospatial data available at 1st Admin Level,Pillar 4: Data Sources,Dimension 4.3,geospatial data,Indicator data availability at sub-national le...,Our source for this indicator is Open Data Wat...,https://odin.opendatawatch.com/
8,SPI.D2.2.Metadata.available,48.3,67.1,18.8,18.8,Metadata Available Score,Pillar 2: Data Services,Dimension 2.2,Online access,Metadata Available Score,Our source for this indicator is Open Data Wat...,https://odin.opendatawatch.com/
19,SPI.D3.16.INST,85.7,71.4,-14.3,14.3,GOAL 16: Peace and Justice Strong Institutions,Pillar 3: Data Products,Dimension 3.16,SDG Goal 16,SDG Goal 16 data availability. Source: UN Glo...,Fraction of Indicators in Goal 16 with value p...,https://unstats.un.org/sdgs/unsdg
28,SPI.D3.9.INDY,100.0,90.0,-10.0,10.0,"GOAL 9: Industry, Innovation and Infrastructure",Pillar 3: Data Products,Dimension 3.9,SDG Goal 9,SDG Goal 9 data availability. Source: UN Glob...,Fraction of Indicators in Goal 9 with value pr...,https://unstats.un.org/sdgs/unsdg
21,SPI.D3.2.HNGR,70.0,60.0,-10.0,10.0,GOAL 2: Zero Hunger,Pillar 3: Data Products,Dimension 3.2,SDG Goal 2,SDG Goal 2 data availability. Source: UN Glob...,Fraction of Indicators in Goal 2 with value pr...,https://unstats.un.org/sdgs/unsdg
18,SPI.D3.15.LAND,83.3,91.7,8.4,8.4,GOAL 15: Life on Land,Pillar 3: Data Products,Dimension 3.15,SDG Goal 15,SDG Goal 15 data availability. Source: UN Glo...,Fraction of Indicators in Goal 15 with value p...,https://unstats.un.org/sdgs/unsdg


## 9. Reconstructing the World Bank fixed-weight formula in Python

In [46]:
# P1
P1 = [
    "SPI.D1.5.POV",
    "SPI.D1.5.CHLD.MORT",
    "SPI.D1.5.DT.TDS.DPPF.XP.ZS",
    "SPI.D1.5.SAFE.MAN.WATER",
    "SPI.D1.5.LFP"
]

# P2
P2_D21 = ["SPI.D2.1.GDDS"]
P2_D22 = ["SPI.D2.2.Openness.subscore"]
P2_D24 = ["SPI.D2.4.NADA"]

# P3
P3 = list(P3_FIELDS.values())

# P4
P4_CENSUS = ["SPI.D4.1.1.POPU", "SPI.D4.1.2.AGRI", "SPI.D4.1.3.BIZZ"]
P4_SURVEY = [
    "SPI.D4.1.4.HOUS", "SPI.D4.1.5.AGSVY", "SPI.D4.1.6.LABR",
    "SPI.D4.1.7.HLTH", "SPI.D4.1.8.BZSVY"
]
EXPECTED_P4_ADMIN = ["SPI.D4.2.3.CRVS"]
P4_ADMIN = sorted([c for c in indicator_cols if c.startswith("SPI.D4.2")])
P4_GEO = ["SPI.D4.3.GEO.first.admin.level"]

check_true("9. Fixed weights", "P4 administrative schema pinned v4.0 bilan kutilganidek",
           P4_ADMIN == EXPECTED_P4_ADMIN, observed=P4_ADMIN, expected=EXPECTED_P4_ADMIN)

def strict_mean(row, cols):
    vals = pd.to_numeric(row[cols], errors="coerce")
    if vals.isna().any():
        return np.nan
    return float(vals.mean())

def fixed_weight_strict(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    active = weights != 0
    if np.isnan(values[active]).any() or np.isnan(weights[active]).any():
        return np.nan
    total = weights.sum()
    if total == 0:
        return np.nan
    return float(np.sum((weights / total) * values))

def recalc_spi(row):
    p1 = fixed_weight_strict([row[c] for c in P1], [1/5]*5)

    d21 = strict_mean(row, P2_D21)
    d22 = strict_mean(row, P2_D22)
    d24 = strict_mean(row, P2_D24)
    p2 = fixed_weight_strict([d21,d22,d24], [1/3]*3)

    p3 = fixed_weight_strict([row[c] for c in P3], [1/16]*16)

    cen = strict_mean(row, P4_CENSUS)
    svy = strict_mean(row, P4_SURVEY)
    adm = strict_mean(row, P4_ADMIN)
    geo = strict_mean(row, P4_GEO)
    p4 = fixed_weight_strict([cen,svy,adm,geo], [1/4]*4)

    d52 = fixed_weight_strict([row[c] for c in P5_D52], [1/10]*10)
    # P5 conceptual dimensions 5.1 / 5.2 / 5.5: amaldagi weights 0 / 1 / 0
    p5 = fixed_weight_strict([0.0,d52,0.0], [0.0,1.0,0.0])

    overall = fixed_weight_strict([p1,p2,p3,p4,p5], [1/5]*5)

    return pd.Series({
        "P1_recalc":p1*100, "P2_recalc":p2*100, "P3_recalc":p3*100,
        "P4_recalc":p4*100, "P5_recalc":p5*100, "SPI_recalc":overall*100
    })

recalc = uzb.apply(recalc_spi, axis=1)
recalc.insert(0, "date", uzb_core["date"].values)

published = uzb_core[[
    "date","SPI.INDEX.PIL1","SPI.INDEX.PIL2","SPI.INDEX.PIL3",
    "SPI.INDEX.PIL4","SPI.INDEX.PIL5","SPI.INDEX"
]].reset_index(drop=True)

recalc_audit = published.merge(recalc, on="date")
for i in range(1,6):
    recalc_audit[f"P{i}_diff"] = recalc_audit[f"P{i}_recalc"] - recalc_audit[f"SPI.INDEX.PIL{i}"]
recalc_audit["SPI_diff"] = recalc_audit["SPI_recalc"] - recalc_audit["SPI.INDEX"]

display(recalc_audit.round(12))

for i in range(1,6):
    d = float(recalc_audit[f"P{i}_diff"].abs().max())
    check_close("9. Fixed weights", f"P{i} recomputation matches the World Bank result", d, 0.0, atol=1e-10)

check_close("9. Fixed weights", "Full recomputation of overall SPI matches the World Bank result",
            float(recalc_audit["SPI_diff"].abs().max()), 0.0, atol=1e-10)


✅ PASS: P4 administrative schema pinned v4.0 bilan kutilganidek
   observed=['SPI.D4.2.3.CRVS'] | expected=['SPI.D4.2.3.CRVS']


,date,SPI.INDEX.PIL1,SPI.INDEX.PIL2,SPI.INDEX.PIL3,SPI.INDEX.PIL4,SPI.INDEX.PIL5,SPI.INDEX,P1_recalc,P2_recalc,P3_recalc,P4_recalc,P5_recalc,SPI_recalc,P1_diff,P2_diff,P3_diff,P4_diff,P5_diff,SPI_diff
0,2023,90.0,76.633333,85.04375,63.283333,95.0,81.992083,90.0,76.633333,85.04375,63.283333,95.0,81.992083,0.0,0.0,0.0,0.0,0.0,0.0
1,2024,90.0,79.133333,80.11250,68.208333,90.0,81.490833,90.0,79.133333,80.11250,68.208333,90.0,81.490833,0.0,0.0,0.0,0.0,0.0,0.0


✅ PASS: P1 qayta hisobi World Bank natijasiga mos
   observed=0.0 | expected=0.0
✅ PASS: P2 qayta hisobi World Bank natijasiga mos
   observed=0.0 | expected=0.0
✅ PASS: P3 qayta hisobi World Bank natijasiga mos
   observed=0.0 | expected=0.0
✅ PASS: P4 qayta hisobi World Bank natijasiga mos
   observed=0.0 | expected=0.0
✅ PASS: P5 qayta hisobi World Bank natijasiga mos
   observed=0.0 | expected=0.0
✅ PASS: Overall SPI to‘liq qayta hisobi World Bank natijasiga mos
   observed=0.0 | expected=0.0


'PASS'

## 10. Verifying that the formulas are present in the pinned World Bank R code

In [47]:
r_lines = r_text.splitlines()

R_PATTERNS = {
    "P5 dim_5_1 weight = 0": "dim_5_1 <- 0",
    "P5 dim_5_2 weight = 1": "dim_5_2 <- 1",
    "P5 dim_5_5 weight = 0": "dim_5_5 <- 0",
    "LOCF ishlatiladi": "na.locf",
    "P4 dimension 4.2 rowMeans": "SPI.DIM4.2.INDEX=rowMeans",
    "P4 fixed formula": "SPI.INDEX.PIL4=(",
    "P5 fixed formula": "SPI.INDEX.PIL5=(",
    "Overall fixed formula": "SPI.INDEX=(pillar_1",
}

r_matches = []
for check_name, pattern in R_PATTERNS.items():
    hits = [(i+1, line.strip()) for i,line in enumerate(r_lines) if pattern in line]
    ok = len(hits) > 0
    log_check("10. R code", check_name, "PASS" if ok else "FAIL",
              observed=len(hits), expected=">=1",
              detail=f"pattern: {pattern}")
    if hits:
        line_no, line = hits[0]
        r_matches.append({"Check":check_name, "Line":line_no, "Pinned_R_excerpt":line[:240]})

r_code_evidence = pd.DataFrame(r_matches)
display(r_code_evidence)


✅ PASS: P5 dim_5_1 weight = 0
   observed=1 | expected=>=1
   pattern: dim_5_1 <- 0
✅ PASS: P5 dim_5_2 weight = 1
   observed=1 | expected=>=1
   pattern: dim_5_2 <- 1
✅ PASS: P5 dim_5_5 weight = 0
   observed=1 | expected=>=1
   pattern: dim_5_5 <- 0
✅ PASS: LOCF ishlatiladi
   observed=5 | expected=>=1
   pattern: na.locf
✅ PASS: P4 dimension 4.2 rowMeans
   observed=1 | expected=>=1
   pattern: SPI.DIM4.2.INDEX=rowMeans
✅ PASS: P4 fixed formula
   observed=1 | expected=>=1
   pattern: SPI.INDEX.PIL4=(
✅ PASS: P5 fixed formula
   observed=1 | expected=>=1
   pattern: SPI.INDEX.PIL5=(
✅ PASS: Overall fixed formula
   observed=1 | expected=>=1
   pattern: SPI.INDEX=(pillar_1


,Check,Line,Pinned_R_excerpt
0,P5 dim_5_1 weight = 0,1283,dim_5_1 <- 0
1,P5 dim_5_2 weight = 1,1284,dim_5_2 <- 1
2,P5 dim_5_5 weight = 0,1285,dim_5_5 <- 0
3,LOCF ishlatiladi,1323,"mutate(across(starts_with(""SPI""), na.locf, na...."
4,P4 dimension 4.2 rowMeans,1392,SPI.DIM4.2.INDEX=rowMeans(across(starts_with('...
5,P4 fixed formula,1428,SPI.INDEX.PIL4=(
6,P5 fixed formula,1434,SPI.INDEX.PIL5=(
7,Overall fixed formula,1438,SPI.INDEX=(pillar_1/pillar_total)*SPI.INDEX.PI...


## 11. Audit boundaries — limitations disclosed openly to reviewers

This section is deliberately kept. The strength of an audit lies not only in its `PASS` results, but in **openly showing which conclusions cannot be proven from this source**.

In [48]:
limitations = pd.DataFrame([
    {
        "Item":"P5 / CPIBY matematik sababi",
        "Status":"PASS",
        "What_SPI_GitHub_proves":"2023=1.0, 2024=0.5; together with the other D5.2 components, the P5 95\u219290 change can be recomputed.",
        "What_still_needs_external_source":"Why CPIBY scores 0.5 \u2014 requires IMF and National Statistics Committee CPI metadata/methodology."
    },
    {
        "Item":"P3 pasaygan SDG maqsadlari",
        "Status":"PASS",
        "What_SPI_GitHub_proves":"Shows precisely which SDG goal scores changed 2023\u21922024.",
        "What_still_needs_external_source":"Har bir pasayishning ildiz sababi — UN SDG Global Database va revisions/availability auditi."
    },
    {
        "Item":"2026 population and agricultural census facts",
        "Status":"WARN",
        "What_SPI_GitHub_proves":"2024 baholash davridagi SPI census score.",
        "What_still_needs_external_source":"2026 dates, population counts, budget, and results \u2014 official national sources/UNFPA and others."
    },
    {
        "Item":"Byudjet, muddat va kelajak reyting prognozlari",
        "Status":"WARN",
        "What_SPI_GitHub_proves":"Joriy SPI ballari va arifmetik sensitivity hisoblari.",
        "What_still_needs_external_source":"Kelajak reytingi, xarajat va muddatlar — alohida scenario/model, smeta va institutsional tasdiq."
    },
])
display(status_style(limitations))

for _, r in limitations.iterrows():
    if r["Status"]=="WARN":
        log_check("11. Scope", r["Item"], "WARN",
                  observed=r["What_SPI_GitHub_proves"],
                  expected=r["What_still_needs_external_source"])


,Item,Status,What_SPI_GitHub_proves,What_still_needs_external_source
0,P5 / CPIBY matematik sababi,PASS,"2023=1.0, 2024=0.5; boshqa D5.2 komponentlari bilan birga P5 95→90 bo‘lganini qayta hisoblash mumkin.",Nega CPIBY scoring 0.5 bo‘lgani — IMF va Milliy statistika qo‘mitasi CPI metadata/metodologiyasi.
1,P3 pasaygan SDG maqsadlari,PASS,Qaysi SDG goal score 2023→2024 o‘zgarganini aniq ko‘rsatadi.,Har bir pasayishning ildiz sababi — UN SDG Global Database va revisions/availability auditi.
2,2026-yil aholi va qishloq xo‘jaligi ro‘yxatga olish faktlari,WARN,2024 baholash davridagi SPI census score.,"2026 sanalari, aholi soni, byudjet va natijalar — milliy rasmiy manbalar/UNFPA va boshqalar."
3,"Byudjet, muddat va kelajak reyting prognozlari",WARN,Joriy SPI ballari va arifmetik sensitivity hisoblari.,"Kelajak reytingi, xarajat va muddatlar — alohida scenario/model, smeta va institutsional tasdiq."


⚠️ WARN: 2026-yil aholi va qishloq xo‘jaligi ro‘yxatga olish faktlari
   observed=2024 baholash davridagi SPI census score. | expected=2026 sanalari, aholi soni, byudjet va natijalar — milliy rasmiy manbalar/UNFPA va boshqalar.
⚠️ WARN: Byudjet, muddat va kelajak reyting prognozlari
   observed=Joriy SPI ballari va arifmetik sensitivity hisoblari. | expected=Kelajak reytingi, xarajat va muddatlar — alohida scenario/model, smeta va institutsional tasdiq.


## 12. FINAL AUDIT SUMMARY — discrepancies are visible here at a glance

In [49]:
audit_log_df = pd.DataFrame(AUDIT_LOG)
claims_audit_df = pd.DataFrame(CLAIM_AUDIT)

# Claim FAILlarni audit logdan tashqari ham hisoblaymiz
claim_fail_count = int((claims_audit_df["Status"]=="FAIL").sum())
check_fail_count = int((audit_log_df["Status"]=="FAIL").sum())
warn_count = int((audit_log_df["Status"]=="WARN").sum())
total_fail = claim_fail_count + check_fail_count

summary = pd.DataFrame([
    ["Pinned release", RELEASE_TAG],
    ["Pinned commit", REF_SHA],
    ["Run UTC", RUN_UTC],
    ["Project numerical claims", len(claims_audit_df)],
    ["Raqamli claim FAIL", claim_fail_count],
    ["Texnik/metodologik FAIL", check_fail_count],
    ["WARN", warn_count],
    ["Total FAIL", total_fail],
], columns=["Audit_item","Value"])
display(summary)

if total_fail == 0:
    display(HTML("""
    <div style="padding:16px;border:2px solid #2e7d32;background:#e8f5e9;border-radius:8px">
      <b>\u2705 NO DISCREPANCIES FOUND IN THE FIGURES CHECKED AGAINST THE PINNED WORLD BANK SPI v4.0.</b><br>
      WARN statuses indicate that a causal claim, or a claim outside the scope of the SPI GitHub source, requires an additional source.
    </div>
    """))
else:
    display(HTML(f"""
    <div style="padding:16px;border:2px solid #c62828;background:#ffebee;border-radius:8px">
      <b>❌ {total_fail} TA FAIL ANIQLANDI.</b><br>
      Do not finalize the report without correcting the red tables below.
    </div>
    """))

if claim_fail_count:
    print("\n\u274c Discrepancies in the project\u2019s figures:")
    display(status_style(claims_audit_df.loc[claims_audit_df["Status"]=="FAIL"]))

if check_fail_count:
    print("\n❌ Texnik/metodologik nomuvofiqliklar:")
    display(status_style(audit_log_df.loc[audit_log_df["Status"]=="FAIL"]))

print("\n\u26a0\ufe0f WARN cases requiring an additional source:")
display(status_style(audit_log_df.loc[audit_log_df["Status"]=="WARN"]))


,Audit_item,Value
0,Pinned release,v4.0
1,Pinned commit,2dd02f746cb4e2159281adc74d6865f948e90ce1
2,Run UTC,2026-09-10T09:36:00.117410+00:00
3,Loyiha raqamli claimlari,315
4,Raqamli claim FAIL,2
5,Texnik/metodologik FAIL,0
6,WARN,4
7,Jami FAIL,2



❌ Loyiha raqamlaridagi nomuvofiqliklar:


,Report_location,Metric,Year,Expected_in_report,Actual_raw_WB,Actual_at_report_precision,Decimals,Difference_at_report_precision,Status,Pinned_source_field,Note
191,9-jadval,Norway — P4,2024,88.200000,88.150000,88.100000,1,-0.100000,FAIL,SPI.INDEX.PIL4,
255,9-jadval,Norway — P4,2024,88.200000,88.150000,88.100000,1,-0.100000,FAIL,SPI.INDEX.PIL4,



⚠️ Qo‘shimcha manba talab qiladigan WARN holatlari:


,Section,Check,Status,Observed,Expected,Detail
11,4.2 P2,SDDS bo‘yicha 95–96 ball natija faqat shartli sensitivity hisobidir,WARN,95.800000,rasmiy prognoz emas,World Bank GitHub kelajakdagi ballni kafolatlamaydi.
13,4.5 P5,CPIBY nima sababdan 0.5 bo‘lgani SPI GitHub bilan yolg‘iz isbotlanmaydi,WARN,score change confirmed,cause requires IMF/NSC metadata,Hujjatda sababni [Gipoteza] yoki qo‘shimcha manba bilan berish kerak.
34,11. Scope,2026-yil aholi va qishloq xo‘jaligi ro‘yxatga olish faktlari,WARN,2024 baholash davridagi SPI census score.,"2026 sanalari, aholi soni, byudjet va natijalar — milliy rasmiy manbalar/UNFPA va boshqalar.",
35,11. Scope,"Byudjet, muddat va kelajak reyting prognozlari",WARN,Joriy SPI ballari va arifmetik sensitivity hisoblari.,"Kelajak reytingi, xarajat va muddatlar — alohida scenario/model, smeta va institutsional tasdiq.",


## 13. Generating the electronic evidence package

This cell produces the following:
- `project_claims_audit.csv` — a PASS/FAIL register of the report's numerical claims;
- `audit_log.csv` — a log of technical and methodological checks;
- `source_hashes.csv` — SHA-256 fingerprints of the pinned source files;
- `audit_manifest.json` — audit version, commit, timestamp, and environment information;
- `AUDIT_SUMMARY.html` — a short electronic audit summary;
- `raw_pinned/` — copies of the World Bank files used in the audit;
- `UZB_SPI_v4_0_EVIDENCE_PACK.zip` — a bundle of all the evidence above.

**Note:** if the separately prepared `UZB_SPI_PROJECT_CLAIMS_REGISTER_FINAL.xlsx` file is kept alongside this notebook, it serves as the numerical register for the report's key claims. The notebook does not depend on Excel generation, so no additional Excel libraries need to be installed in Colab.

In [50]:
# -------------------------------------------------------
# 13.1 CSV va JSON audit fayllari
# -------------------------------------------------------
claims_audit_df.to_csv(EVIDENCE_DIR/"project_claims_audit.csv", index=False, encoding="utf-8-sig")
audit_log_df.to_csv(EVIDENCE_DIR/"audit_log.csv", index=False, encoding="utf-8-sig")
hash_manifest.to_csv(EVIDENCE_DIR/"source_hashes.csv", index=False, encoding="utf-8-sig")

manifest = {
    "title": "Uzbekistan SPI v4.0 reproducible audit",
    "report_version": REPORT_VERSION,
    "world_bank_release_tag": RELEASE_TAG,
    "pinned_commit_sha": REF_SHA,
    "release_url": f"{REPO_PAGE}/releases/tag/{RELEASE_TAG}",
    "run_utc": RUN_UTC,
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "country_iso3": COUNTRY_ISO,
    "target_years": TARGET_YEARS,
    "claim_checks": int(len(claims_audit_df)),
    "claim_fail_count": int((claims_audit_df["Status"]=="FAIL").sum()),
    "technical_fail_count": int((audit_log_df["Status"]=="FAIL").sum()),
    "warn_count": int((audit_log_df["Status"]=="WARN").sum()),
    "source_files": HASH_ROWS,
}
(EVIDENCE_DIR/"audit_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

# -------------------------------------------------------
# 13.2 HTML audit xulosasi
# -------------------------------------------------------
html_path = EVIDENCE_DIR/"AUDIT_SUMMARY.html"
failed_claims_html = (
    claims_audit_df.loc[claims_audit_df["Status"]=="FAIL"].to_html(index=False)
    if (claims_audit_df["Status"]=="FAIL").any()
    else "<p><b>Raqamli FAIL topilmadi.</b></p>"
)
warn_html = audit_log_df.loc[audit_log_df["Status"]=="WARN"].to_html(index=False)

html_doc = f"""<!doctype html>
<html lang='uz'><head><meta charset='utf-8'><title>UZB SPI v4.0 Audit</title>
<style>
body{{font-family:Arial,sans-serif;max-width:1200px;margin:30px auto;line-height:1.45;color:#1f2937}}
table{{border-collapse:collapse;width:100%;font-size:13px}} th,td{{border:1px solid #d1d5db;padding:7px;vertical-align:top}}
th{{background:#f3f4f6}} .ok{{background:#ecfdf5;padding:14px;border-left:5px solid #059669}}
.bad{{background:#fef2f2;padding:14px;border-left:5px solid #dc2626}}
code{{background:#f3f4f6;padding:2px 5px;border-radius:4px}}
</style></head><body>
<h1>Uzbekistan SPI 2023\u20132024 \u2014 Reproducible Audit</h1>
<p><b>Release:</b> {RELEASE_TAG}<br><b>Commit:</b> <code>{REF_SHA}</code><br><b>Run UTC:</b> {RUN_UTC}</p>
<div class="{'ok' if total_fail==0 else 'bad'}"><b>Total FAIL: {total_fail}</b></div>
<h2>Raqamli nomuvofiqliklar</h2>{failed_claims_html}
<h2>WARN cases requiring an additional source</h2>{warn_html}
<h2>Source files and SHA-256</h2>{hash_manifest.to_html(index=False)}
</body></html>"""
html_path.write_text(html_doc, encoding="utf-8")

# -------------------------------------------------------
# 13.3 README va ZIP
# -------------------------------------------------------
readme = f"""UZBEKISTAN SPI v4.0 AUDIT EVIDENCE PACK
================================================
Report version: {REPORT_VERSION}
World Bank release: {RELEASE_TAG}
Pinned commit: {REF_SHA}
Run UTC: {RUN_UTC}

Talqin:
- PASS: the report figure matches the pinned World Bank v4.0 value.
- FAIL: the figure or recalculation does not match.
- WARN: an additional official source is required for a causal or external claim.

Fayllar:
- project_claims_audit.csv
- audit_log.csv
- source_hashes.csv
- audit_manifest.json
- AUDIT_SUMMARY.html
- raw_pinned/
"""
(EVIDENCE_DIR/"README.txt").write_text(readme, encoding="utf-8")

zip_base = Path("UZB_SPI_v4_0_EVIDENCE_PACK")
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=EVIDENCE_DIR))

print("✅ Elektron dalil paketi tayyor.")
print("HTML:", html_path.resolve())
print("ZIP :", zip_path.resolve())
print("CSV :", (EVIDENCE_DIR/"project_claims_audit.csv").resolve())


✅ Elektron dalil paketi tayyor.
HTML: /content/UZB_SPI_AUDIT_EVIDENCE/AUDIT_SUMMARY.html
ZIP : /content/UZB_SPI_v4_0_EVIDENCE_PACK.zip
CSV : /content/UZB_SPI_AUDIT_EVIDENCE/project_claims_audit.csv


### 13.4. Downloading the ZIP file in Google Colab — optional

Setting `DOWNLOAD_ZIP = True` opens Colab's download dialog. Download is disabled by default under **Run all** so the notebook does not hang waiting for a browser prompt.

In [51]:
DOWNLOAD_ZIP = False

if DOWNLOAD_ZIP:
    try:
        from google.colab import files
        files.download("UZB_SPI_v4_0_EVIDENCE_PACK.zip")
    except ImportError:
        print("Bu Google Colab emas. ZIP joriy papkada saqlangan.")
    except Exception as e:
        print("Yuklab olish oynasi ochilmadi:", e)
        print("Fayl baribir yaratildi: UZB_SPI_v4_0_EVIDENCE_PACK.zip")
else:
    print("Automatic download is disabled. Set DOWNLOAD_ZIP=True if needed.")


Avtomatik download o‘chirilgan. Kerak bo‘lsa DOWNLOAD_ZIP=True qiling.


## 14. Optional diagnostic: comparing the pinned v4.0 against the current `master`

**The audit verdict does not depend on this section.** It exists solely to check whether the World Bank has since revised historical rows.

Set `COMPARE_MASTER = True` to run it.

In [52]:
COMPARE_MASTER = False

if not COMPARE_MASTER:
    print("Master comparison disabled. The pinned audit result is unchanged.")
else:
    master_url = "https://raw.githubusercontent.com/worldbank/SPI/master/03_output_data/SPI_index.csv"
    try:
        master_bytes = fetch_bytes(master_url, "current master SPI_index.csv")
        spi_master, _ = decode_csv(master_bytes)

        pinned = spi.loc[
            (spi["iso3c"]==COUNTRY_ISO)&(spi["date"].isin(TARGET_YEARS)), CORE_COLS
        ].sort_values("date").reset_index(drop=True)
        current = spi_master.loc[
            (spi_master["iso3c"]==COUNTRY_ISO)&(spi_master["date"].isin(TARGET_YEARS)), CORE_COLS
        ].sort_values("date").reset_index(drop=True)

        display(pd.concat({"PINNED_v4.0":pinned, "CURRENT_master":current}, axis=1))
        numeric = [c for c in CORE_COLS if c.startswith("SPI.")]
        same = (
            len(pinned)==len(current)==2
            and np.allclose(pinned[numeric], current[numeric], equal_nan=True, atol=1e-12)
        )
        if same:
            print("✅ Current master va pinned v4.0 UZB 2023/2024 uchun bir xil.")
        else:
            print("\u26a0\ufe0f Current master differs from pinned v4.0. Use the PINNED result for the audit.")
    except Exception as e:
        print("Master diagnostikasi bajarilmadi:", e)
        print("This does not affect the primary pinned audit.")


Master taqqoslash o‘chirilgan. Pinned audit natijasi o‘zgarmaydi.


## 15. How to use this audit in the report

1. For numerical claims in the final scientific-practical report, a `PASS` result constitutes the primary numerical evidence.
2. A `WARN` result does not mean the number is wrong; it means the causal explanation or external fact must be separately confirmed against IMF, UN, ILOSTAT, official national sources, or other relevant sources.
3. Global ranks are recomputed by sorting, in descending order, all economies with a non-missing `SPI.INDEX` value in 2024.
4. The Chapter 8 values 84.49 → 87.82 → 89.85 are not an official forecast; they are conditional scenarios based on the SPI's actual aggregation formulas.
5. The audit verdict is always based on the pinned **v4.0 / commit `2dd02f7...`**; comparison with `master` is for diagnostic purposes only.